# Bias, Variance & Regularization — AI Lab Instructor Notebook

*Evaluation & Debugging · Medium*

Build interview-grade intuition for under/overfitting, bias/variance tradeoffs, and regularization (L1/L2, early stopping), grounded in experiments.

**Outcome.** Students can:
- diagnose bias vs variance from learning curves,
- choose regularization knobs and justify them,
- explain L1 vs L2 and their effects on weights/features,
- reason about model capacity and data size.

6 tasks, 1 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Bias, Variance & Regularization — Student Lab (Breast Cancer)

You will diagnose under/overfitting using capacity sweeps and learning curves, then connect that to regularization knobs.

In [ ]:
# No source solution — left as an exercise.

## Task 2: Logistic regression baseline
*Section: Baseline*

## Section 1 — Baseline

### Task 1.1: Logistic regression baseline

- Fit a logistic regression with standardization
- Report train and val accuracy

**Checkpoint:** Why is logistic regression usually lower-variance than deep trees?

In [ ]:
# Baseline logistic regression
lr = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
lr.fit(Xtr, ytr)
print('train_acc', accuracy_score(ytr, lr.predict(Xtr)))
print('val_acc', accuracy_score(yva, lr.predict(Xva)))

# Rationale: linear model is constrained capacity; typically lower variance than deep trees.

**Why this works.** linear model is constrained capacity; typically lower variance than deep trees.

## Task 3: Depth sweep
*Section: Capacity sweep (Decision Tree depth)*

## Section 2 — Capacity sweep (Decision Tree depth)

### Task 2.1: Depth sweep

- For depths = 1..15, fit a DecisionTreeClassifier(max_depth=d)
- Track train_acc and val_acc
- Print the best depth by val accuracy

**Checkpoint:** Which depths look high-bias? Which look high-variance?

In [ ]:
# Depth sweep
depths = list(range(1, 16))
train_acc = []
val_acc = []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=0)
    clf.fit(Xtr, ytr)
    train_acc.append(accuracy_score(ytr, clf.predict(Xtr)))
    val_acc.append(accuracy_score(yva, clf.predict(Xva)))
best_i = int(np.argmax(val_acc))
print('best_depth', depths[best_i], 'val_acc', val_acc[best_i])

# Rationale: as depth increases, bias decreases but variance increases; look for widening train-val gap.

**Why this works.** as depth increases, bias decreases but variance increases; look for widening train-val gap.

## Task 4: Sweep C for L2 logistic regression
*Section: Regularization sweep (LogReg C)*

## Section 3 — Regularization sweep (LogReg C)

### Task 3.1: Sweep C for L2 logistic regression

- Try C in [0.01, 0.1, 1, 10, 100]
- Compare train vs val accuracy

**Checkpoint:** What does decreasing C do to bias/variance?
<img src="/images/w5-d3-bias-variance-accuracy-curves.svg" alt="Bias-variance with train/validation accuracy vs C" width="640" />

Visual intuition:
- Left (low C = strong regularization): bias dominates (underfitting risk).
- Middle: best tradeoff region (lowest validation/test error).
- Right (high C = weak regularization): variance dominates (overfitting risk).

- C axis uses log scale: `[0.01, 0.1, 1, 10, 100]`.

In [ ]:
# C sweep (L2)
Cs = [0.01, 0.1, 1.0, 10.0, 100.0]
for C in Cs:
    lrC = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000, C=C, penalty='l2'))])
    lrC.fit(Xtr, ytr)
    tr = accuracy_score(ytr, lrC.predict(Xtr))
    va = accuracy_score(yva, lrC.predict(Xva))
    print('C', C, 'train', tr, 'val', va)

# Rationale: smaller C => stronger regularization => higher bias, lower variance.

**Why this works.** smaller C => stronger regularization => higher bias, lower variance.

## Section 3 - Regularization Strength (C)

- High C -> less regularization -> model more complex, lower bias, potentially high variance (overfitting).
- Low C -> more regularization -> model simpler, higher bias, potentially lower variance (underfitting).

## Task 5: L1 vs L2 sparsity
*Section: Regularization sweep (LogReg C)*

### Task 3.2: L1 vs L2 sparsity

- Fit L1 logistic regression (solver=liblinear)
- Count how many coefficients are exactly 0
- Compare to L2

**FAANG Gotcha:** correlated features can make L1 unstable (it picks one arbitrarily).

In [ ]:
# L1 vs L2 sparsity
lr_l2 = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000, C=1.0, penalty='l2'))])
lr_l2.fit(Xtr, ytr)
w_l2 = lr_l2.named_steps['model'].coef_.ravel()

lr_l1 = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000, C=1.0, penalty='l1', solver='liblinear'))])
lr_l1.fit(Xtr, ytr)
w_l1 = lr_l1.named_steps['model'].coef_.ravel()

print('l2 zeros', int(np.sum(np.isclose(w_l2, 0.0))))
print('l1 zeros', int(np.sum(np.isclose(w_l1, 0.0))))

# Rationale: L1 encourages sparsity; correlated features can cause arbitrary selection.

**Why this works.** L1 encourages sparsity; correlated features can cause arbitrary selection.

## Task 6: Train with increasing data sizes
*Section: Learning curve by subsampling*

## Section 4 — Learning curve by subsampling

### Task 4.1: Train with increasing data sizes

- For fractions in [0.1, 0.2, 0.4, 0.6, 0.8, 1.0], train logistic regression on that fraction
- Track train vs val accuracy

**Interview Angle:** How do you decide whether to get more data vs regularize more?

Use this with Section 3 curve intuition for bias/variance diagnosis from train-vs-val gaps.

In [ ]:
# Learning curve by subsampling
fracs = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
idx_all = np.arange(len(Xtr))
for f in fracs:
    k = max(5, int(f * len(Xtr)))
    idx = rng.choice(idx_all, size=k, replace=False)
    Xsub, ysub = Xtr[idx], ytr[idx]
    lr_sub = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000, C=1.0))])
    lr_sub.fit(Xsub, ysub)
    tr = accuracy_score(ysub, lr_sub.predict(Xsub))
    va = accuracy_score(yva, lr_sub.predict(Xva))
    print('frac', f, 'n', k, 'train', tr, 'val', va)

# Rationale: if both train/val are low -> bias; if train high, val low -> variance; more data helps variance.

**Why this works.** if both train/val are low -> bias; if train high, val low -> variance; more data helps variance.

## Section 4 - Learning Curve by Subsampling

> ð¡ If both train/validation are low, suspect **bias**. If train is high and validation is low, suspect **variance**. More data usually helps variance. If validation accuracy flattens or worsens despite more data, increase regularization.